In [12]:
import pandas as pd

df = pd.read_hdf(
    "metr-la.h5",
    key="df"
)

data = df.values

print(data.shape)

(34272, 207)


In [13]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(data)

print(data_scaled.shape)

(34272, 207)


In [14]:
import numpy as np

SEQ_LEN = 12

X = []
y = []

for i in range(len(data_scaled)-SEQ_LEN):

    X.append(
        data_scaled[i:i+SEQ_LEN]
    )

    y.append(
        data_scaled[i+SEQ_LEN]
    )


X = np.array(X)
y = np.array(y)


print(X.shape)
print(y.shape)

(34260, 12, 207)
(34260, 207)


In [15]:
train_size = int(len(X)*0.7)
val_size = int(len(X)*0.1)


X_train = X[:train_size]
y_train = y[:train_size]


X_val = X[
    train_size:
    train_size+val_size
]

y_val = y[
    train_size:
    train_size+val_size
]


X_test = X[
    train_size+val_size:
]

y_test = y[
    train_size+val_size:
]


print(X_train.shape)
print(X_test.shape)

(23982, 12, 207)
(6852, 12, 207)


In [16]:
import torch

X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)


X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

In [17]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader


train_dataset = TensorDataset(
    X_train,
    y_train
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)

In [18]:
import torch.nn as nn


class TemporalConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )


    def forward(self,x):

        return torch.relu(
            self.conv(x)
        )

In [19]:
class AdaptiveGraphConv(nn.Module):

    def __init__(self, num_nodes, channels):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self, x):

        adaptive_adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adaptive_adj,
            x
        )

        x = x.permute(
            0,2,3,1
        )

        x = self.weight(x)

        x = x.permute(
            0,3,1,2
        )

        return torch.relu(x)

In [20]:
class AdaptiveSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = TemporalConv(
            1,
            32
        )

        self.graph = AdaptiveGraphConv(
            num_nodes=207,
            channels=32
        )

        self.temp2 = TemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.graph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        x = x.squeeze(-1)

        return x

In [21]:
model = AdaptiveSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 207])
torch.Size([64, 207])


In [22]:
model = AdaptiveSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [23]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0


    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()


        predictions = model(
            X_batch
        )

        prediction_loss = criterion(
            predictions,
            y_batch
        )

        physics_loss = torch.mean(
            (
                predictions[:,1:]
                -
                predictions[:,:-1]
            )**2
        )

        loss = (
            prediction_loss
            +
            0.01 * physics_loss
        )


        loss.backward()


        optimizer.step()


        total_loss += loss.item()


    avg_loss = total_loss / len(train_loader)


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {avg_loss:.6f}"
    )

Epoch 1/30 Loss: 0.040068
Epoch 2/30 Loss: 0.017176
Epoch 3/30 Loss: 0.015643
Epoch 4/30 Loss: 0.014841
Epoch 5/30 Loss: 0.013717
Epoch 6/30 Loss: 0.013089
Epoch 7/30 Loss: 0.012551
Epoch 8/30 Loss: 0.012133
Epoch 9/30 Loss: 0.011729
Epoch 10/30 Loss: 0.011419
Epoch 11/30 Loss: 0.011100
Epoch 12/30 Loss: 0.010789
Epoch 13/30 Loss: 0.010510
Epoch 14/30 Loss: 0.010262
Epoch 15/30 Loss: 0.010033
Epoch 16/30 Loss: 0.009840
Epoch 17/30 Loss: 0.009674
Epoch 18/30 Loss: 0.009549
Epoch 19/30 Loss: 0.009435
Epoch 20/30 Loss: 0.009325
Epoch 21/30 Loss: 0.009247
Epoch 22/30 Loss: 0.009186
Epoch 23/30 Loss: 0.009136
Epoch 24/30 Loss: 0.009098
Epoch 25/30 Loss: 0.009064
Epoch 26/30 Loss: 0.009034
Epoch 27/30 Loss: 0.009012
Epoch 28/30 Loss: 0.008984
Epoch 29/30 Loss: 0.008963
Epoch 30/30 Loss: 0.008945


In [24]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np


test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.04462367
RMSE: 0.09504872
